# Actualización de Tablas Vida

## 1. Importación de paqueterías

In [ ]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text
from sqlalchemy.types import NVARCHAR, Float, Integer, DateTime, DECIMAL, Date
import pandas as pd
from sqlalchemy import create_engine, text
import urllib


## 2. Conexión a SQL Server

In [ ]:

# -----------------------------
# 1️⃣ Conexión a SQL Server Azure
# -----------------------------

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=ikaldb.database.windows.net;"
    "DATABASE=actuarial;"
    "UID=CloudSA0052c2f7;"
    "PWD=ricostamales01#;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)
engine_azure = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)
print("Conexión a Azure SQL establecida.")

# -----------------------------
# 2️⃣ Conexión a SQL Server Local
# -----------------------------
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=IKAL14\\SQLEXPRESS;"
    "DATABASE=db;"
    "Trusted_Connection=yes;"
)
engine_local = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)
print("Conexión a SQL Server Local establecida.")


Conexión a Azure SQL establecida.


## 3. Carga de datos

In [ ]:
# -----------------------------
# 2️⃣ Leer Excel con datos nuevos
# -----------------------------
archivo_excel = r"C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/2025/202509/202509_Siniestros_Marine_PROCESADO"
df_nuevo = pd.read_excel(archivo_excel)
print(f"Datos del Excel '{archivo_excel}' leídos correctamente.")

Datos del Excel 'C:/Users/KOT14/OneDrive - Kot Insurance Company AG/Vida/Consolidado_2026/v2_marzo.xlsx' leídos correctamente.


## 4. Limpieza y carga de tabla vida

In [ ]:
# -----------------------------
# 3️⃣ Limpiar tabla 'vida'
# -----------------------------
with engine_azure.begin() as conn:
    conn.execute(text("TRUNCATE TABLE actuarial.dbo.vida"))
print("Tabla 'vida' limpiada correctamente.")

with engine_local.begin() as conn:
    conn.execute(text("TRUNCATE TABLE actuarial.dbo.vida"))
print("Tabla 'vida' limpiada correctamente.")

#with engine_azure.begin() as conn:
#    conn.execute(text("TRUNCATE TABLE actuarial.dbo.vidamaestra"))
#print("Tabla 'vidamaestra' limpiada correctamente en azure.")
#with engine_local.begin() as conn:
#    conn.execute(text("TRUNCATE TABLE actuarial.dbo.vidamaestra"))
#print("Tabla 'vidamaestra' limpiada correctamente en local.")
# -----------------------------

# -----------------------------
# 4️⃣ Subir nuevos datos a 'vida'
# -----------------------------
print("📊 Actualizando tabla 'vida' (mes actual)...")
df_nuevo.to_sql(
    'vida',
    con=engine_azure,
    if_exists='replace',
    index=False,
    schema='dbo'
)
df_nuevo.to_sql(
    'vida',
    con=engine_local,
    if_exists='replace',
    index=False,
    schema='dbo'
)
print(f"✅ Tabla 'vida' reemplazada con {len(df_nuevo)} registros")
# -----------------------------


Tabla 'vida' limpiada correctamente.
📊 Actualizando tabla 'vida' (mes actual)...
✅ Tabla 'vida' reemplazada con 57860 registros


## 5. Inserción y actualización de tabla Vidamaestra

In [ ]:
# =============================================================================
# 5️⃣ INSERCIÓN Y ACTUALIZACIÓN DE TABLA VIDAMAESTRA CON MERGE
# =============================================================================

# 📋 PASO 1: Obtener estructura de tabla SQL existente
print("\n" + "="*80)
print("PASO 1: Obtener estructura de tabla SQL")
print("="*80)

# Obtener columnas que existen en vidamaestra
query_columnas = """
SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = 'vidamaestra' AND TABLE_SCHEMA = 'dbo'
ORDER BY ORDINAL_POSITION
"""
columnas_sql = set(pd.read_sql(query_columnas, con=engine_azure)['COLUMN_NAME'].tolist())
columnas_sql = set(pd.read_sql(query_columnas, con=engine_local)['COLUMN_NAME'].tolist())
print(f"✓ Columnas en tabla SQL (vidamaestra): {len(columnas_sql)}")

# Obtener columnas del Excel
columnas_excel = set(df_nuevo.columns.tolist())
print(f"✓ Columnas en Excel: {len(columnas_excel)}")

# Encontrar columnas válidas (que existen en ambas)
columnas_validas = sorted(list(columnas_excel & columnas_sql))
print(f"✓ Columnas válidas para MERGE: {len(columnas_validas)}")

# Mostrar columnas que están en Excel pero no en SQL (estas se ignoran)
solo_excel = columnas_excel - columnas_sql
if solo_excel:
    print(f"⚠️  Columnas en Excel pero NO en SQL ({len(solo_excel)}): {sorted(list(solo_excel))}")

# 📋 PASO 2: Preparar datos del Excel
print("\n" + "="*80)
print("PASO 2: Preparar datos del Excel")
print("="*80)

# Mantener solo las columnas válidas
df_vida_limpio = df_nuevo[columnas_validas].copy()
print(f"✓ DataFrame filtrado: {len(df_vida_limpio)} registros × {len(columnas_validas)} columnas")

# Definir columnas clave para el MERGE (PK de vidamaestra)
columnas_clave = ['SINIESTRO', 'FICHA', 'POLIZA']
print(f"✓ Columnas clave (PK): {columnas_clave}")

# Contar registros antes del merge
registros_antes_query = "SELECT COUNT(*) as total FROM vidamaestra"
registros_antes = pd.read_sql(registros_antes_query, con=engine_azure)['total'][0]
registros_antes = pd.read_sql(registros_antes_query, con=engine_local)['total'][0]
print(f"✓ Registros en vidamaestra ANTES del merge: {registros_antes:,}")

# 📋 PASO 3: Cargar datos a tabla temporal
print("\n" + "="*80)
print("PASO 3: Cargar datos a tabla temporal")
print("="*80)

df_vida_limpio.to_sql('temp_vida_mes', con=engine_local, if_exists='replace', index=False, schema='dbo')
df_vida_limpio.to_sql('temp_vida_mes', con=engine_azure, if_exists='replace', index=False, schema='dbo')
print(f"✓ Tabla temporal 'temp_vida_mes' creada con {len(df_vida_limpio):,} registros")

# 📋 PASO 4: Limpiar registros con RFC incorrecto
print("\n" + "="*80)
print("PASO 4: Limpiar registros con RFC incorrecto")
print("="*80)

cleanup_query = """
DELETE vm
FROM vidamaestra vm
INNER JOIN temp_vida_mes t
  ON vm.SINIESTRO = t.SINIESTRO
  AND vm.FICHA = t.FICHA
  AND vm.POLIZA = t.POLIZA
WHERE vm.RFC != t.RFC
  AND LEN(vm.RFC) != 13  -- RFC viejo está mal
  AND LEN(t.RFC) = 13    -- RFC nuevo está correcto
"""

with engine_azure.begin() as conn:
    result = conn.execute(text(cleanup_query))
    registros_limpiados = result.rowcount
    if registros_limpiados > 0:
        print(f"✅ Registros con RFC incorrecto eliminados en Azure: {registros_limpiados:,}")
    else:
        print(f"ℹ️  No se encontraron registros para limpiar en Azure")

with engine_local.begin() as conn:
    result = conn.execute(text(cleanup_query))
    registros_limpiados = result.rowcount
    if registros_limpiados > 0:
        print(f"✅ Registros con RFC incorrecto eliminados en Local: {registros_limpiados:,}")
    else:
        print(f"ℹ️  No se encontraron registros para limpiar en Local")

# 📋 PASO 5: Construir y ejecutar MERGE
print("\n" + "="*80)
print("PASO 5: Ejecutar MERGE (actualización/inserción)")
print("="*80)

# Construcción de cláusulas MERGE
on_clause = ' AND '.join([f'target.[{col}] = source.[{col}]' for col in columnas_clave])
insert_cols = ', '.join([f'[{col}]' for col in columnas_validas])
insert_vals = ', '.join([f'source.[{col}]' for col in columnas_validas])

# Construir cláusula UPDATE (excluir columnas clave)
update_cols = [col for col in columnas_validas if col not in columnas_clave]
update_set = ', '.join([f'target.[{col}] = source.[{col}]' for col in update_cols])

# MERGE con OUTPUT para rastrear operaciones
merge_query = f"""
MERGE vidamaestra AS target
USING temp_vida_mes AS source
ON ({on_clause})

WHEN MATCHED THEN
    UPDATE SET {update_set}

WHEN NOT MATCHED THEN
    INSERT ({insert_cols})
    VALUES ({insert_vals})

OUTPUT $action AS Action;
"""

# Ejecutar MERGE
with engine_local.begin() as conn:
    result = conn.execute(text(merge_query))
    merge_output = result.fetchall()

with engine_azure.begin() as conn:
    result = conn.execute(text(merge_query))
    merge_output = result.fetchall()

# Analizar resultados del MERGE
updates = sum(1 for row in merge_output if row[0] == 'UPDATE')
inserts = sum(1 for row in merge_output if row[0] == 'INSERT')

print(f"✅ MERGE completado:")
print(f"   • Registros actualizados: {updates:,}")
print(f"   • Registros nuevos (insertados): {inserts:,}")

# 📋 PASO 6: Marcar registros inactivos
print("\n" + "="*80)
print("PASO 6: Marcar registros inactivos (que ya no están en vida actual)")
print("="*80)

marca_inactivos_query = """
UPDATE vidamaestra
SET ACTIVO = 0,
    FECHA_BAJA = GETDATE()
WHERE NOT EXISTS (
    SELECT 1 FROM temp_vida_mes t
    WHERE vidamaestra.SINIESTRO = t.SINIESTRO
      AND vidamaestra.FICHA = t.FICHA
      AND vidamaestra.RFC = t.RFC
      AND vidamaestra.POLIZA = t.POLIZA
)
AND ACTIVO = 1  -- Solo marcar los que actualmente están activos
"""

with engine_azure.begin() as conn:
    result = conn.execute(text(marca_inactivos_query))
    marcados = result.rowcount
    if marcados > 0:
        print(f"✅ Registros marcados como inactivos en Azure: {marcados:,}")
    else:
        print(f"ℹ️  No hay registros para marcar como inactivos en Azure")

with engine_local.begin() as conn:
    result = conn.execute(text(marca_inactivos_query))
    marcados = result.rowcount
    if marcados > 0:
        print(f"✅ Registros marcados como inactivos en Local: {marcados:,}")
    else:
        print(f"ℹ️  No hay registros para marcar como inactivos en Local")

# 📋 PASO 7: Resumen final
print("\n" + "="*80)
print("RESUMEN FINAL")
print("="*80)

# Contar después del merge
registros_despues_query = "SELECT COUNT(*) as total FROM vidamaestra WHERE ACTIVO = 1"
registros_despues_azure = pd.read_sql(registros_despues_query, con=engine_azure)['total'][0]
registros_despues_local = pd.read_sql(registros_despues_query, con=engine_local)['total'][0]

print(f"\n✅ ESTADÍSTICAS:")
print(f" • Registros en vidamaestra ANTES: {registros_antes:,}")
print(f" • Registros limpiados (RFC incorrecto): {registros_limpiados:,}")
print(f" • Registros procesados del Excel: {len(df_vida_limpio):,}")
print(f" • Registros ACTUALIZADOS en vidamaestra: {updates:,}")
print(f" • Registros NUEVOS insertados: {inserts:,}")
print(f" • Registros marcados INACTIVOS: {marcados:,}")
print(f" • Registros ACTIVOS en vidamaestra DESPUÉS en Azure: {registros_despues_azure:,}")
print(f" • Registros ACTIVOS en vidamaestra DESPUÉS en Local: {registros_despues_local:,}")

# Limpiar tabla temporal
with engine_local.begin() as conn:
    conn.execute(text("DROP TABLE temp_vida_mes"))
print(f"\n🧹 Tabla temporal 'temp_vida_mes' eliminada en Local")
print("="*80)

with engine_azure.begin() as conn:
    conn.execute(text("DROP TABLE temp_vida_mes"))
print(f"\n🧹 Tabla temporal 'temp_vida_mes' eliminada en Azure")
print("="*80)


PASO 1: Obtener estructura de tabla SQL
✓ Columnas en tabla SQL (vidamaestra): 46
✓ Columnas en Excel: 44
✓ Columnas válidas para MERGE: 43
⚠️  Columnas en Excel pero NO en SQL (1): ['CAUSA FALLECIMIENTO ORIGINAL']

PASO 2: Preparar datos del Excel
✓ DataFrame filtrado: 57860 registros × 43 columnas
✓ Columnas clave (PK): ['SINIESTRO', 'FICHA', 'POLIZA']
✓ Registros en vidamaestra ANTES del merge: 57,883

PASO 3: Cargar datos a tabla temporal
✓ Tabla temporal 'temp_vida_mes' creada con 57,860 registros

PASO 4: Limpiar registros con RFC incorrecto
ℹ️  No se encontraron registros para limpiar

PASO 5: Ejecutar MERGE (actualización/inserción)
✅ MERGE completado:
   • Registros actualizados: 57,860
   • Registros nuevos (insertados): 0

PASO 6: Marcar registros inactivos (que ya no están en vida actual)
ℹ️  No hay registros para marcar como inactivos

RESUMEN FINAL

✅ ESTADÍSTICAS:
 • Registros en vidamaestra ANTES: 57,883
 • Registros limpiados (RFC incorrecto): 0
 • Registros procesado